# Lưu TEST cache từ đúng Output181
**Accelerator None, Internet ON**, bật Secret `codex` của ducanh2403. Không cần Add Input/checkpoint. Tải đúng181, kiểm10SHA và vectors258x512 rồi tạo Dataset Private mới. Không chạy lại cache/model; không dùng latest. Sau khi Dataset xử lý xong, chia sẻ quyền cho Ngân và gửi path mount.


In [ ]:
from pathlib import Path
OWNER='ducanh2403'
SLUG='csa-prompt-test-cache-v181'
TITLE='CSA prompt TEST cache v181'
SECRET_NAME='codex'
DESTINATION=Path('/kaggle/working/csa_prompt_test_cache_v181_dataset')


In [ ]:
import json,os,re,subprocess,sys,tempfile,shutil
from kaggle_secrets import UserSecretsClient
if not re.fullmatch(r'[a-zA-Z0-9_-]+',OWNER) or not re.fullmatch(r'[a-z0-9]+(?:-[a-z0-9]+)*',SLUG):raise ValueError('Invalid OWNER/SLUG')
if not 6<=len(SLUG)<=50 or not 6<=len(TITLE)<=50:raise ValueError('Title/slug must have6–50characters')
if DESTINATION.exists():raise RuntimeError('Preserve existing output; choose fresh DESTINATION')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<4*1024**3:raise RuntimeError('Need4GiB free for download/copy; no files removed')
try:token=UserSecretsClient().get_secret(SECRET_NAME).strip()
except Exception:raise RuntimeError('Enable the named Secret for the owner of private output181') from None
if not token:raise RuntimeError('Empty API token')
def run_cli(args,authenticated=True):
    environment=os.environ.copy()
    for k in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):environment.pop(k,None)
    if authenticated:environment['KAGGLE_API_TOKEN']=token
    environment['PYTHONUNBUFFERED']='1'
    process=subprocess.Popen(list(map(str,args)),env=environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace')
    for line in process.stdout:print(line.replace(token,'[REDACTED]'),end='',flush=True)
    if process.wait():raise RuntimeError('API command failed; preserve staged files and read redacted log above')
CLI=[sys.executable,'-c','from kaggle.cli import main; main()']
run_cli([sys.executable,'-m','pip','install','-q','kaggle==2.2.5'],authenticated=False)


In [ ]:
VERSION_PREFLIGHT="\nimport tempfile\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi=KaggleApi()\nfor version in ('181',):\n    with tempfile.TemporaryDirectory() as directory,patch.object(api,'build_kaggle_client') as factory:\n        endpoint=factory.return_value.__enter__.return_value.kernels.kernels_api_client.list_kernel_session_output\n        endpoint.return_value=SimpleNamespace(files=[],next_page_token=None,log=None)\n        api.kernels_output('ducanh2403/2-vi-cai-kia-qua-nhieu/'+version,path=directory)\n        request=endpoint.call_args.args[0]\n        assert request.version_label=='v'+version and request.user_name=='ducanh2403' and request.kernel_slug=='2-vi-cai-kia-qua-nhieu','Exact version ignored'\nprint('Exact181 API routing preflight PASS; no network in this check.',flush=True)\n"
run_cli([sys.executable,'-u','-c',VERSION_PREFLIGHT])


In [ ]:
"""Byte-preserving CPU transport for exact169 cache and exact161 weights; no models."""
import hashlib
import json
from pathlib import Path
import re
import shutil
import numpy as np


def sha(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream,'sha256').hexdigest()


def pattern(names):
    return r'(^|/)('+'|'.join(re.escape(n) for n in names)+r')$'


def locate(root,name,expected):
    matches=[p for p in Path(root).rglob(name) if p.is_file() and sha(p)==expected]
    if len(matches)!=1:
        raise ValueError(f'Need one exact {name}; found {len(matches)} matching SHA, no version fallback')
    return matches[0]


def package_full_cache(download,destination,inventory):
    """Preserve exact175 fresh cache, without caption weights or neural imports."""
    destination=Path(destination);files=inventory['cache_files']
    if inventory['cache_version']!='ducanh2403/2-vi-cai-kia-qua-nhieu/175':raise ValueError('Expected exact175')
    if destination.exists():raise FileExistsError('Preserve existing destination')
    if any(Path(n).name!=n or '/' in n or '\\' in n for n in files):raise ValueError('Unsafe payload name')
    root=locate(download,'cache_receipt.json',files['cache_receipt.json']).parent
    for n,h in files.items():
        if not (root/n).is_file() or sha(root/n)!=h:raise ValueError('Missing/changed175 payload:'+n)
    read=lambda n:json.loads((root/n).read_text(encoding='utf-8'))
    receipt=read('cache_receipt.json');manifest=read('cache_manifest.json');state=read('run_status.json')
    expected={'cohort.json','prompt_texts.json','prompt_links.jsonl','summary.json','review_examples.json','prompt_embeddings.npy','mapping_receipt.json','reuse_report.json','cache_manifest.json'}
    if receipt['status']!='FULL_PROMPT_TEXT_CACHE_COMPLETE' or set(receipt['payload_sha256'])!=expected or any(files.get(n)!=h for n,h in receipt['payload_sha256'].items()):raise ValueError('Full cache receipt contract differs')
    if state!=dict(complete=True,training=False,test_used=False,caption_weights_loaded=False):raise ValueError('Wrong cache run status')
    if manifest['status']!='FULL_PROMPT_TEXT_CACHE_COMPLETE' or manifest['cache_mode']!='fresh_all' or manifest['shape']!=[410,512] or manifest['training'] or manifest['test_used']:raise ValueError('Wrong full cache manifest')
    texts=read('prompt_texts.json');cohort=read('cohort.json');vectors=np.load(root/'prompt_embeddings.npy',allow_pickle=False)
    if len(texts)!=410 or texts!=sorted(set(texts)) or any(not isinstance(t,str) or not t.strip() for t in texts):raise ValueError('Wrong full text vocabulary')
    if vectors.shape!=(410,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError('Embedding shape/dtype/norm differs')
    if len(cohort['train'])!=len(set(cohort['train'])) or len(cohort['train'])!=113287 or len(cohort['val'])!=len(set(cohort['val'])) or len(cohort['val'])!=5000 or cohort['test'] or set(cohort['train'])&set(cohort['val']):raise ValueError('Full cohort coverage differs')
    destination.mkdir(parents=True)
    for n,h in files.items():
        shutil.copy2(root/n,destination/n)
        if sha(destination/n)!=h:raise ValueError('Copied bytes changed:'+n)
    report=dict(status='FULL_PROMPT_V175_PRESERVED',cache_source=inventory['cache_version'],files_sha256=files,
        source_review_zip_sha256=inventory['source_review_zip_sha256'],original_bytes=True,embedding_shape=[410,512],
        models_executed=False,training=False,test_used=False)
    (destination/'transport_receipt.json').write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
    return report


def package(cache_download,weight_download,destination,inventory):
    destination=Path(destination)
    if destination.exists():raise FileExistsError('Preserve existing destination; choose a fresh one')
    files=inventory['cache_files']
    if any(Path(n).name!=n or '/' in n or '\\' in n for n in files):raise ValueError('Unsafe payload name')
    receipt_path=locate(cache_download,'cache_receipt.json',files['cache_receipt.json'])
    root=receipt_path.parent
    receipt=json.loads(receipt_path.read_text(encoding='utf-8'))
    if receipt['status']!='PROMPT_TEXT_CACHE_COMPLETE':raise ValueError('Incomplete cache')
    if any(n not in files or files[n]!=h for n,h in receipt['payload_sha256'].items()):raise ValueError('Cache receipt identities differ')
    if set(receipt['payload_sha256'])!={'prompt_embeddings.npy','prompt_texts.json','prompt_links.jsonl','cohort.json','summary.json','cache_manifest.json'}:
        raise ValueError('Cache receipt payload coverage differs')
    for n,h in files.items():
        if not (root/n).is_file() or sha(root/n)!=h:raise ValueError('Missing/changed169 payload:'+n)
    state=json.loads((root/'run_status.json').read_text(encoding='utf-8'))
    if not state['complete'] or state['training'] or state['test_used']:raise ValueError('Wrong cache run status')
    vectors=np.load(root/'prompt_embeddings.npy',allow_pickle=False)
    if vectors.shape!=(318,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):
        raise ValueError('Embedding shape/dtype/norm differs')
    weight=locate(weight_download,'best.bin',inventory['checkpoint_sha256'])
    # Binary identity is checked without deserializing weights or importing Torch.
    destination.mkdir(parents=True)
    for n in files:shutil.copy2(root/n,destination/n)
    shutil.copy2(weight,destination/'csa_ce161_best.bin')
    expected={**files,'csa_ce161_best.bin':inventory['checkpoint_sha256']}
    for n,h in expected.items():
        if sha(destination/n)!=h:raise ValueError('Copied bytes changed:'+n)
    report=dict(status='PROMPT_PILOT_INPUTS_PRESERVED',cache_source=inventory['cache_version'],
        checkpoint_source=inventory['checkpoint_version'],files_sha256=expected,
        original_bytes=True,models_executed=False,training=False,test_used=False)
    (destination/'transport_receipt.json').write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
    return report


In [ ]:
import numpy as np
INVENTORY={'cache_version': 'ducanh2403/2-vi-cai-kia-qua-nhieu/181', 'cache_files': {'test_regions.npz': '7fb385e1a03c5241c4d65cc240d0434e041284a9a829198c5167f0725be57b1a', 'test_metadata.json': '4271b46b7ab67117433bfdf84f6eb9ba66515ada9b3c4ae4d2e507bc2cede018', 'vocabulary.json': '924be210aa0daa8b0968a6d69ad2c66b3de603f49c35f4aa67247c75d6173fc7', 'test_receipt.json': '7a212068c7e73f6f0791906c1ec81d56e3178fd62317bdf0a5e528350d95055d', 'prompt_links.jsonl': '54e25cd17c9842c07fb4ef0b3da4ed4ad276571d3711d51bf0b3ac1a86cc3c55', 'prompt_texts.json': 'afdad5d42734adfbd38883f1fa7a22aaba1b76ee8cd7510635fd3e8b22d44553', 'prompt_embeddings.npy': '7119776b70380376a343dd9130bed6d2e5fc19594d83cf4bdd724a7343a4fb35', 'summary.json': '3c9b863b73d9efdeda7f5979d25c829c74d805505a856c965141dc275a0e8847', 'source_coverage.json': 'b5f66a8bcb40027dcdd11b0d71c92551d68a11f91a6daf56e1157cdd0b5a7a3c', 'receipt.json': 'f86708aa99a3a07087a55164cd3fa099d01e01eef15a91da38540dc28a5c3742'}, 'source_review_zip_sha256': '44f752a96e5fd228f704f45a0cc7de0160b97a4636ad958ec4c59a2e77b1de8a'}
DOWNLOAD=Path(tempfile.mkdtemp(prefix='prompt-test181-',dir='/tmp'))
run_cli(CLI+['kernels','output',INVENTORY['cache_version'],'-p',str(DOWNLOAD),'--file-pattern',pattern(INVENTORY['cache_files'])])
root=locate(DOWNLOAD,'receipt.json',INVENTORY['cache_files']['receipt.json']).parent
for name,digest in INVENTORY['cache_files'].items():
    if Path(name).name!=name or not (root/name).is_file() or sha(root/name)!=digest:raise ValueError('Missing/changed181 payload:'+name)
receipt=json.loads((root/'receipt.json').read_text(encoding='utf-8'))
if receipt['status']!='PROMPT_TEST_PREPARATION_COMPLETE' or any(INVENTORY['cache_files'].get(n)!=h for n,h in receipt['payload_sha256'].items()):raise ValueError('Wrong cache receipt')
vectors=np.load(root/'prompt_embeddings.npy',allow_pickle=False)
if vectors.shape!=(258,512) or vectors.dtype!=np.float32 or not np.isfinite(vectors).all() or not np.allclose(np.linalg.norm(vectors,axis=1),1,atol=2e-6,rtol=0):raise ValueError('Wrong text vectors')
rows=[json.loads(line) for line in (root/'prompt_links.jsonl').read_text(encoding='utf-8').splitlines()]
if len(rows)!=5000 or len({r['coco_id'] for r in rows})!=5000 or any(r['group']!='test' for r in rows):raise ValueError('Wrong TEST5000 prompt coverage')
DESTINATION.mkdir(parents=True)
for name,digest in INVENTORY['cache_files'].items():
    shutil.copy2(root/name,DESTINATION/name)
    if sha(DESTINATION/name)!=digest:raise ValueError('Copied bytes changed:'+name)
(DESTINATION/'transport_receipt.json').write_text(json.dumps(dict(status='EXACT181_PRESERVED',source=INVENTORY['cache_version'],files_sha256=INVENTORY['cache_files'],models_executed=False),indent=2),encoding='utf-8')
print('All10 payloads verified and copied unchanged; TEST5000/258vectors. No model executed.',flush=True)


In [ ]:
CREATE_DATASET_SCRIPT="import sys\nfrom kaggle.api.kaggle_api_extended import KaggleApi\napi = KaggleApi()\napi.authenticate()\nresult = api.dataset_create_new(sys.argv[1], public=False, quiet=False,\n                                convert_to_csv=False, dir_mode='skip')\nif result is None or str(getattr(result, 'status', '')).lower() != 'ok' or getattr(result, 'error', None):\n    raise RuntimeError('Dataset creation rejected: ' + str(getattr(result, 'error', 'Missing success response')))\nprint('Dataset creation accepted by API:', result.url, flush=True)\n"

metadata_path=DESTINATION/'dataset-metadata.json'
if metadata_path.exists():raise RuntimeError('Preserve existing metadata; use fresh destination')
metadata=dict(title=TITLE,id=f'{OWNER}/{SLUG}',licenses=[dict(name='other')],
    description='Private exact181 TEST5000 ROI and structured prompt cache, frozen CLIP vectors. No caption weights or training. Upstream data/model terms retained.')
try:
    metadata_path.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    run_cli([sys.executable,'-u','-c',CREATE_DATASET_SCRIPT,str(DESTINATION)])
finally:metadata_path.unlink(missing_ok=True)
print('API accepted creation; processing may still be active. Check page before retrying.',flush=True)
print(f'https://www.kaggle.com/datasets/{OWNER}/{SLUG}',flush=True)
print('After processing, grant Ngan access in Dataset sharing; do not give anyone your API token.',flush=True)
